In [2]:
import pandas as pd
from dotenv import load_dotenv
import os
import requests
import polars as pl
from io import StringIO
import logging
from zephyr_etl.sources.co_dot import CoDOTParser

load_dotenv()

CDOT_API_KEY = os.getenv('CDOT_API_KEY')
CDOT_WEATHER_URL = 'https://data.cotrip.org/api/v1/weatherStations'


In [18]:
response.json()['features'][1]

{'type': 'Feature',
 'geometry': {'coordinates': [-104.63169, 37.41726],
  'type': 'Point',
  'srid': 4326},
 'properties': {'name': '025N03330RWS1RHS : 0.8 miles S of I-25BL/CR 60 in Lynn',
  'sensors': [{'name': 'Sensor-514480',
    'id': 'Sensor-514480',
    'type': 'temperature',
    'currentReading': '22.30'},
   {'name': 'Sensor-514488',
    'id': 'Sensor-514488',
    'type': 'average wind speed',
    'currentReading': '2.60'},
   {'name': 'Sensor-514489',
    'id': 'Sensor-514489',
    'type': 'average wind direction',
    'currentReading': 'E'},
   {'name': 'Sensor-514519',
    'id': 'Sensor-514519',
    'type': 'road surface sensor error',
    'currentReading': 'none'},
   {'name': 'Sensor-514498',
    'id': 'Sensor-514498',
    'type': 'precipitation rate',
    'currentReading': '0.00'},
   {'name': 'Sensor-514517',
    'id': 'Sensor-514517',
    'type': 'road surface friction index',
    'currentReading': '0.82'},
   {'name': 'Sensor-514500',
    'id': 'Sensor-514500',
    '

In [7]:
response = requests.get(
    "https://data.cotrip.org/api/v1/weatherStations",
    params={'apiKey': CDOT_API_KEY, 'limit': 1000},  # High limit
    timeout=30
)
direct_results = response.json()['features']

In [9]:
len(direct_results)


172

In [3]:
import logging
import sys

# Configure root logger
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s',
    handlers=[logging.StreamHandler(sys.stdout)],
    force=True  # Force reconfiguration
)

In [ ]:
import pandas as pd

# Download EPA data
url = "https://www.fueleconomy.gov/feg/epadata/vehicles.csv"
df = pd.read_csv(url)

# Analyze by vehicle class
avg_mpg_by_class = df.groupby(['year', 'VClass'])['comb08'].mean()

# Track SUV efficiency improvements
suv_trend = df[df['VClass'].str.contains('SUV')].groupby('year')['comb08'].mean()

In [ ]:
avg_mpg_by_class

In [4]:
data = CoDOTParser(api_key=CDOT_API_KEY).fetch_met_data()

2025-08-01 09:37:58,467 - zephyr_etl.sources.co_dot - INFO - Requesting data from: https://data.cotrip.org/api/v1/weatherStations
2025-08-01 09:38:11,873 - zephyr_etl.sources.co_dot - INFO - Fetched 172 stations in 13.41 seconds


In [5]:
len(data)

172

In [6]:
data

[{'type': 'Feature',
  'geometry': {'coordinates': [-106.985183, 39.330905],
   'type': 'Point',
   'srid': 4326},
  'properties': {'name': '082E02670RWS1SWC @ Snowmass Creek Rd/CR-11 CAM',
   'sensors': [{'name': 'Sensor-519564',
     'id': 'Sensor-519564',
     'type': 'road surface sensor type',
     'currentReading': 'contact passive'},
    {'name': 'Sensor-519566',
     'id': 'Sensor-519566',
     'type': 'road surface type',
     'currentReading': 'concrete'},
    {'name': 'Sensor-519574',
     'id': 'Sensor-519574',
     'type': 'road surface sensor error',
     'currentReading': 'other'},
    {'name': 'Sensor-519575',
     'id': 'Sensor-519575',
     'type': 'road surface sensor type',
     'currentReading': 'other'},
    {'name': 'Sensor-519576',
     'id': 'Sensor-519576',
     'type': 'road surface status',
     'currentReading': '1'},
    {'name': 'Sensor-519577',
     'id': 'Sensor-519577',
     'type': 'road surface type',
     'currentReading': 'asphalt'},
    {'name': '

In [ ]:
df = pl.DataFrame(data).unnest('properties')

In [ ]:
df

In [ ]:
response.json()['features']

In [ ]:
df = pl.DataFrame(response.json()['features'])
df

In [ ]:
df.head(1).unnest(['geometry','properties'])

In [ ]:
print(df.select(pl.col("geometry")).schema)
print(df.select(pl.col("properties")).schema)

In [ ]:
df_unnested = df.select([
    # Keep the type column
    pl.col("type"),
    
    # Unnest geometry to get coordinates
    pl.col("geometry").struct.field("coordinates").alias("coordinates"),
    pl.col("geometry").struct.field("type").alias("geometry_type"),
    pl.col("geometry").struct.field("srid").alias("srid"),
    
    # Unnest all properties fields
    pl.col("properties").struct.field("*")
])

In [ ]:
df_unnested.explode('sensors')

In [ ]:
df.unnest("properties").describe()

In [ ]:
import json
df = pl.from_dict(response.json())
df[2,1]['properties']['publicName']

In [ ]:
df

In [ ]:
df[2,1]['properties'].keys()

In [ ]:
test_data = pl.DataFrame(response.json()['features'])[0,2]

In [ ]:
test_data

In [ ]:
test_data['name']